<a href="https://colab.research.google.com/github/ckstrouse/CA5-/blob/main/CA05_kNN_Movie_Recommender.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CA05 – kNN Based Movie Recommender Engine

## Overview
In this notebook we build a **k-Nearest Neighbors (kNN) Recommender Engine** for movies.  
Given a query movie – *The Post* – the engine will find the **5 most similar movies** in the dataset based on genre features and IMDB ratings.

This mimics the "More Like This" feature found on platforms such as Netflix or IMDb.

## Step 1 – Import Libraries
We import `pandas` for data handling, `numpy` for numerical operations, and `sklearn` for the kNN model.

In [19]:
# Import required libraries
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors

print("Libraries imported successfully.")

Libraries imported successfully.


## Step 2 – Load the Dataset
The dataset contains **30 movies** with the following columns:
- `Movie ID`, `Movie Name`
- `IMDB Rating`
- Genre flags (binary): `Biography`, `Drama`, `Thriller`, `Comedy`, `Crime`, `Mystery`
- `label` (all zeros – not used for classification)

We load it directly from GitHub using the exact URL provided.

In [20]:
# Load the dataset
df = pd.read_csv("https://github.com/ArinB/MSBA-CA-Data/raw/main/CA05/movies_recommendation_data.csv")

print(f"Dataset loaded: {df.shape[0]} rows × {df.shape[1]} columns")
df.head(10)

Dataset loaded: 30 rows × 11 columns


,Movie ID,Movie Name,IMDB Rating,Biography,Drama,Thriller,Comedy,Crime,Mystery,History,Label
0,58,The Imitation Game,8.0,1,1,1,0,0,0,0,0
1,8,Ex Machina,7.7,0,1,0,0,0,1,0,0
2,46,A Beautiful Mind,8.2,1,1,0,0,0,0,0,0
3,62,Good Will Hunting,8.3,0,1,0,0,0,0,0,0
4,97,Forrest Gump,8.8,0,1,0,0,0,0,0,0
5,98,21,6.8,0,1,0,0,1,0,1,0
6,31,Gifted,7.6,0,1,0,0,0,0,0,0
7,3,Travelling Salesman,5.9,0,1,0,0,0,1,0,0
8,51,Avatar,7.9,0,0,0,0,0,0,0,0
9,47,The Karate Kid,7.2,0,1,0,0,0,0,0,0


## Step 3 – Explore the Data
Let's inspect the column names, data types, and check for any missing values.

In [21]:
# Check column names and data types
print("Column names and dtypes:")
print(df.dtypes)
print("\nMissing values per column:")
print(df.isnull().sum())

Column names and dtypes:
Movie ID         int64
Movie Name      object
IMDB Rating    float64
Biography        int64
Drama            int64
Thriller         int64
Comedy           int64
Crime            int64
Mystery          int64
History          int64
Label            int64
dtype: object

Missing values per column:
Movie ID       0
Movie Name     0
IMDB Rating    0
Biography      0
Drama          0
Thriller       0
Comedy         0
Crime          0
Mystery        0
History        0
Label          0
dtype: int64


In [22]:
# Display basic statistics for numerical columns
df.describe()

,Movie ID,IMDB Rating,Biography,Drama,Thriller,Comedy,Crime,Mystery,History,Label
count,30.000000,30.000000,30.000000,30.000000,30.000000,30.000000,30.000000,30.000000,30.000000,30.0
mean,48.133333,7.696667,0.233333,0.600000,0.100000,0.100000,0.133333,0.100000,0.100000,0.0
std,29.288969,0.666169,0.430183,0.498273,0.305129,0.305129,0.345746,0.305129,0.305129,0.0
min,1.000000,5.900000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0
25%,27.750000,7.300000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0
50%,48.500000,7.750000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0
75%,64.250000,8.175000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0
max,98.000000,8.800000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.0


## Step 4 – Prepare the Feature Matrix

The kNN algorithm measures similarity purely on **numerical features**.  
We select the feature columns: `IMDB Rating` and the six binary genre columns.  
We drop `Movie ID`, `Movie Name`, and `label` as they are not features used for distance calculation.

In [23]:
# Define the feature columns used for similarity
# Including 'History' which exists in the dataset and is relevant to The Post
feature_cols = ['IMDB Rating', 'Biography', 'Drama', 'Thriller', 'Comedy', 'Crime', 'Mystery', 'History']

# Extract the feature matrix X
X = df[feature_cols]

print("Feature matrix shape:", X.shape)
X.head()

Feature matrix shape: (30, 8)


,IMDB Rating,Biography,Drama,Thriller,Comedy,Crime,Mystery,History
0,8.0,1,1,1,0,0,0,0
1,7.7,0,1,0,0,0,1,0
2,8.2,1,1,0,0,0,0,0
3,8.3,0,1,0,0,0,0,0
4,8.8,0,1,0,0,0,0,0


## Step 5 – Fit the kNN Model

We use `NearestNeighbors` from `sklearn.neighbors`.  
- `n_neighbors = 6`: we ask for 6 neighbors because the query movie itself will appear as the closest match (distance = 0), so we retrieve 6 and then drop the query itself to get the **top 5 recommendations**.
- `metric = 'euclidean'`: standard Euclidean distance is used to measure similarity in the feature space.
- `algorithm = 'auto'`: sklearn automatically picks the most efficient algorithm (ball_tree, kd_tree, or brute force) based on data size.

In [24]:
# Instantiate and fit the kNN model on the full dataset
# n_neighbors=6 because the query movie itself will be one of the neighbors
knn_model = NearestNeighbors(n_neighbors=6, metric='euclidean', algorithm='auto')
knn_model.fit(X)

print("kNN model fitted on", X.shape[0], "movies with", X.shape[1], "features.")

kNN model fitted on 30 movies with 8 features.


## Step 6 – Define the Query Movie: *The Post*

The user is looking at **"The Post"**.  
Its feature vector, as given in the assignment, is:

| Feature | Value |
|---|---|
| IMDB Rating | 7.2 |
| Biography | 1 (Yes) |
| Drama | 1 (Yes) |
| Thriller | 0 (No) |
| Comedy | 0 (No) |
| Crime | 0 (No) |
| Mystery | 0 (No) |
| History | 1 (Yes) — *not in dataset, not used for kNN* |

> **Note:** The dataset does not contain a `History` column, so History is not included in the feature vector. Similarity is computed only on the 7 features above.

In [25]:
# Define the feature vector for the query movie 'The Post'
# Order must match feature_cols: ['IMDB Rating', 'Biography', 'Drama', 'Thriller', 'Comedy', 'Crime', 'Mystery', 'History']
the_post = pd.DataFrame(
    [[7.2, 1, 1, 0, 0, 0, 0, 1]],  # History = 1 (Yes)
    columns=feature_cols
)

print("Query movie feature vector – The Post:")
print(the_post)

Query movie feature vector – The Post:
   IMDB Rating  Biography  Drama  Thriller  Comedy  Crime  Mystery  History
0          7.2          1      1         0       0      0        0        1


## Step 7 – Find the 5 Most Similar Movies

We call `knn_model.kneighbors()` with the query vector.  
The method returns:
- `distances`: Euclidean distances to the 6 nearest neighbors
- `indices`: Row indices in our dataset corresponding to those neighbors

Since *The Post* is not in the dataset, **all 6 results are genuine recommendations** — but we still display the top 5 as required by the assignment.

In [26]:
# Run kNN query: find 6 nearest neighbors to 'The Post'
distances, indices = knn_model.kneighbors(the_post)

print("Raw neighbor indices:", indices)
print("Raw distances:      ", distances)

Raw neighbor indices: [[28 27 29 16  2  9]]
Raw distances:       [[0.9        1.         1.0198039  1.16619038 1.41421356 1.41421356]]


## Step 8 – Display the Recommendations

We map the returned indices back to the original dataframe to retrieve movie names and their details.  
We display the **Top 5** recommendations with their IMDB ratings, genre tags, and Euclidean distance score.

In [27]:
# Extract the top 5 recommendations (first 5 neighbors)
top5_indices = indices[0][:5]
top5_distances = distances[0][:5]

# Build a results dataframe
recommendations = df.iloc[top5_indices][['Movie Name', 'IMDB Rating', 'Biography', 'Drama',
                                          'Thriller', 'Comedy', 'Crime', 'Mystery']].copy()
recommendations.insert(0, 'Rank', range(1, 6))
recommendations['Distance'] = np.round(top5_distances, 4)
recommendations = recommendations.reset_index(drop=True)

print("="*70)
print("  TOP 5 MOVIE RECOMMENDATIONS FOR: 'The Post'")
print("="*70)
print(recommendations.to_string(index=False))
print("="*70)

  TOP 5 MOVIE RECOMMENDATIONS FOR: 'The Post'
 Rank       Movie Name  IMDB Rating  Biography  Drama  Thriller  Comedy  Crime  Mystery  Distance
    1 12 Years a Slave          8.1          1      1         0       0      0        0    0.9000
    2    Hacksaw Ridge          8.2          1      1         0       0      0        0    1.0000
    3   Queen of Katwe          7.4          1      1         0       0      0        0    1.0198
    4   The Wind Rises          7.8          1      1         0       0      0        0    1.1662
    5 A Beautiful Mind          8.2          1      1         0       0      0        0    1.4142


In [28]:
# Display as a clean formatted table
recommendations

,Rank,Movie Name,IMDB Rating,Biography,Drama,Thriller,Comedy,Crime,Mystery,Distance
0,1,12 Years a Slave,8.1,1,1,0,0,0,0,0.9000
1,2,Hacksaw Ridge,8.2,1,1,0,0,0,0,1.0000
2,3,Queen of Katwe,7.4,1,1,0,0,0,0,1.0198
3,4,The Wind Rises,7.8,1,1,0,0,0,0,1.1662
4,5,A Beautiful Mind,8.2,1,1,0,0,0,0,1.4142


## Step 9 – Interpretation of Results

The table above shows the **5 movies most similar to *The Post*** based on Euclidean distance in the feature space of IMDB rating and genre flags.

- **Distance = 0.0** means the movie is a perfect feature-match (same IMDB rating and identical genre flags).
- Lower distance → higher similarity → better recommendation.
- Movies sharing the `Biography = 1` and `Drama = 1` flags (the dominant traits of *The Post*) naturally float to the top.

Since *The Post* is a **Biographical Drama** with an IMDB rating of 7.2, the recommended movies are those that are most similar along these dimensions. The kNN algorithm does not know about directors, actors, or themes — it only knows the numerical features in the dataset.

In [29]:
print("Interpretation:")
print("The recommended movies are the most similar to 'The Post' based on genre composition and IMDB rating.")
print("Since KNN uses distance, smaller distance means higher similarity.")
print("These movies share similar characteristics such as being drama/biography focused.")

Interpretation:
The recommended movies are the most similar to 'The Post' based on genre composition and IMDB rating.
Since KNN uses distance, smaller distance means higher similarity.
These movies share similar characteristics such as being drama/biography focused.


## Conclusion

We successfully built a **kNN-based Movie Recommender Engine** using Python and scikit-learn. The key steps were:

1. **Loaded** the 30-movie dataset from the UCI/GitHub source.
2. **Selected features**: IMDB Rating + 6 binary genre columns.
3. **Fitted** a `NearestNeighbors` model with Euclidean distance.
4. **Queried** the model with the feature vector for *The Post*.
5. **Retrieved and displayed** the Top 5 most similar movies as recommendations.

This forms the core back-end logic of a real movie recommendation website's "More Like This" section.